In [1]:
# Task 2.1 – Import and Setup [3 marks]
# 1m - Import college database with attendance collection
# 1m - insert all json records
# 1m - close json / ensure collection is cleared before inserting new data (no duplicate)

from pymongo import MongoClient
import json

# Connect to MongoDB
client = MongoClient("mongodb://localhost:27017/")
db = client["college"]
collection = db["attendance"]

# Clear old records
collection.delete_many({})

# Load JSON data
with open("attendance.json", "r") as f:
    records = json.load(f)

f.close() #explicitly close - just for A levels 

# Insert new records
collection.insert_many(records)
print("Imported records into MongoDB successfully.")

Imported records into MongoDB successfully.


In [11]:
for record in collection.find({'civics_class': '25S02X'}, {'name':1, "_id":0}):
    print(record['civics_class'])

KeyError: 'civics_class'

In [4]:

# Task 4.2 – Query Attendance by Date [5 marks]
# 2m - Applies the correct query on specified date and status of "Absent"
# 1m - Tabular formatting of printout
# 1m - Correct output displayed 
# 1m - Counts the total number absent

def pprint(labels, items, width):
    for i in range(len(labels)):
        print(f"{labels[i]:<{width[i]}}", end="")
    print()
    print("-" * sum(width))
    
    for record in items:
        for i, label in enumerate(labels):
            print(f"{record.get(label, ''):<{width[i]}}", end="")
        print()
        
def get_absentees_by_date(date):
    print(f"\nAbsentees on {date}:")
    query = {"date": date, "status": "Absent"}
    results = collection.find(query)
    count = collection.count_documents(query)
    
    pprint(["stu_id", "name", "civics_class", "reason_category"], results, [20, 20, 15, 15])
    print(f"Total absent: {count}")

# Test
get_absentees_by_date("2024-04-16")



Absentees on 2024-04-16:
stu_id              name                civics_class   reason_category
----------------------------------------------------------------------
25092               Chen Wei Li         25S02X         Official Event 
25027               Rahul Reddy         25S02Y         Official Event 
25044               Rajesh Varma        25S02Y         Valid Family   
25061               Chutima Nopparat    25S02Y         Official Event 
Total absent: 4


In [5]:
# Task 2.3 (6marks)
#1m - correct operator applied to start_date ("$gte")
#1m - correct operator applied to end_date ("$tte")
#1m - correct range query applied to start_date and end_date
#1m - logic to group records by class
#1m - to count number absent per class
#1m - output is correct
def num_absentees(sdate, edate):
    print(f"\nNumber of Absentees from {sdate} and {edate}")
    # Get all absent records
    absent_docs = collection.find({"status": "Absent", "date":{"$gte": sdate, "$lte": edate}})
    
    # Tally using Python dict
    summary = {}
    for doc in absent_docs:
        cclass = doc["civics_class"]
        summary[cclass] = summary.get(cclass, 0) + 1

    # Print sorted summary by class group
    for cclass in sorted(summary):
        print(f"{cclass}  {summary[cclass]}")

# Test
num_absentees("2024-04-14", "2024-04-16")



Number of Absentees from 2024-04-14 and 2024-04-16
25S02X  7
25S02Y  9


In [4]:
#Task 2.4: Update attendance for particular student (9m)
'''
Marks
1m - User input integration and value assignment
1m - Defines and uses a query filter (stu_id, date)
1m - ... reports if the student/date do not exist. 
1m - Correct use of $set for "Present" and $unset to clear irrelevant fields
1m - Handles "Absent" case with correct use of $set
1m - Correct conditional logic for "Medical" to include evidence_link
3m - Update is successful for each of the following scenarios: Absent to Present, Present to Absent, Absent with evidence link

'''
def update_attendance(date, stu_id):
    query = {"stu_id": stu_id, "date": date}

    if collection.count_documents(query) > 0:        
        status = input("Enter status: ")
        if status == "Present":
            update = { "$set": {
                            "status": "Present"
                        },
                        "$unset": {
                            "reason_category": "",
                            "remarks": "",
                            "evidence_link": "",
                        }
                     }
            collection.update_one(query, update)
        elif status == "Absent":
            category = input("Enter reason category: ")
            remarks = input("Enter remarks: ")
            if category == "Medical":
                evidence_link = input("Enter file name of evidence: ")
            
                update = {  "$set": {
                                "status": "Absent",
                                "reason_category": category,
                                "remarks": remarks,
                                "evidence_link": evidence_link
                            }
                         }
                collection.update_one(query, update)
            else:
                update = {  "$set": {
                                "status": "Absent",
                                "reason_category": category,
                                "remarks": remarks
                            }
                         }
                collection.update_one(query, update)
        print("Record updated")
        print(f"{collection.find(query)[0]}")
    else:
        print(f"Attendance for student {stu_id} on {date} does not exist.")
            
            
        
update_attendance("2024-04-17", "25021")
            

Enter status:  Absent
Enter reason category:  Valid Family
Enter remarks:  Travel
Record updated
{'_id': ObjectId('682865d97fcee4289c98cc28'), 'stu_id': '25021', 'name': 'Tan Yu Wei', 'civics_class': '25S02X', 'date': '2024-04-17', 'status': 'Absent', 'reason_category': 'Valid Family', 'remarks': 'Travel'}


In [5]:
#Task 2.5 (3 marks)
# 1m - Print Menu
# 1m - Loop to continually choose options, and allow user to quit
# 1m - Gather necessary data and call the correct function. 

print("College XYZ Attendance System")
while True: 
    print("\n1. Print absentees by date")
    print("2. Number of absentees in date range")
    print("3. Update student attendance")
    print("4. Quit")
    choice = int(input("Choose option: "))
    
    if choice == 1:
        date = input("Enter date: ")
        get_absentees_by_date(date)

    elif choice == 2:
        sdate = input("Enter start date: ")
        edate = input("Enter end date: ")
        num_absentees(sdate, edate)
    elif choice == 3: 
        date = input("Enter date: ")
        stu_id = input("Enter student id: ")
        update_attendance(date, stu_id)
    elif choice == 4:
        print("Bye")
        break;
            
    

College XYZ Attendance System

1. Print absentees by date
2. Number of absentees in date range
3. Update student attendance
4. Quit
Choose option:  1
Enter date:  2024-04-16

Absentees on 2024-04-16:
stu_id              name                civics_class   reason_category
----------------------------------------------------------------------
25092               Chen Wei Li         25S02X         Official Event 
25027               Rahul Reddy         25S02Y         Official Event 
25044               Rajesh Varma        25S02Y         Valid Family   
25061               Chutima Nopparat    25S02Y         Official Event 
Total absent: 4

1. Print absentees by date
2. Number of absentees in date range
3. Update student attendance
4. Quit
Choose option:  4
Bye
